# goldset_all_scenarios 전체 LLM 재평가

- **목적**: `goldset_all_scenarios.json`의 1,688개 항목 전체를 LLM judge로 재평가하여 relevance_grade를 갱신한다.

- **기존 대비 변경점**
  - 기존 노트북은 Retriever를 호출하여 후보를 새로 뽑은 뒤 라벨링하는 방식이었으나, 이 노트북은 **이미 goldset에 있는 도서 정보를 그대로 활용**한다.
  - `asyncio.to_thread` + `asyncio.Semaphore`를 사용해 동시에 여러 요청을 처리한다. (순차 처리 대비 약 8배 빠름)

- **입출력**
  - 입력: `evaluation/dataset/goldset_all_scenarios.json`
  - 중간 저장(resume용): `evaluation/dataset/goldset_all_scenarios_reeval.jsonl`
  - 최종 출력: `evaluation/dataset/goldset_all_scenarios_reeval.json`
  - 원본 파일은 수정하지 않는다.

- **테스트 실행**: Cell 2의 `LIMIT = 10` 으로 설정 후 실행
- **전체 실행**: `LIMIT = None` 으로 설정 후 실행 (예상 소요: ~40분)

In [33]:
import os
os.chdir("/home/ubuntu/work/somin/backend")
os.getcwd()

'/home/ubuntu/work/somin/backend'

In [34]:
import json
import uuid
import time
import random
import asyncio
import requests
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.core.config import settings
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 경로 및 설정

In [35]:
CLOVA_API_KEY = settings.CLOVA_API_KEY

REPO_ROOT     = Path("/home/ubuntu/work/somin/")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data_after_retrieval.json"
INPUT_JSON    = DATASET_DIR / "goldset_s01_s02_s03_s08_s15_s17_rerun.json"
OUTPUT_JSONL  = DATASET_DIR / "goldset_s01_s02_s03_s08_s15_s17_reeval3.jsonl"
OUTPUT_JSON   = DATASET_DIR / "goldset_s01_s02_s03_s08_s15_s17_reeval3.json"

URL         = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CONCURRENCY = 8    # 동시 요청 수 (429 발생 시 5로 낮춤)
MAX_RETRIES = 5
LIMIT       = None  # 테스트 시 숫자로 변경 (예: 10), 전체 실행 시 None

print(f"입력 파일: {INPUT_JSON}")
print(f"시나리오 파일: {SCENARIO_PATH}")
print(f"출력 JSONL: {OUTPUT_JSONL}")
print(f"출력 JSON: {OUTPUT_JSON}")
print(f"동시 요청 수: {CONCURRENCY}")
print(f"LIMIT: {LIMIT}")

입력 파일: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_rerun.json
시나리오 파일: /home/ubuntu/work/somin/evaluation/dataset/scenario_data_after_retrieval.json
출력 JSONL: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_reeval3.jsonl
출력 JSON: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_reeval3.json
동시 요청 수: 8
LIMIT: None


# LLM-Judge 설정

In [36]:
SYSTEM_PROMPT = """
당신은 매우 보수적으로 책 추천 적합성을 판단하는 평가자입니다.

입력으로는 다음 두 정보가 주어집니다.

1. request: 사용자의 검색 요청 전체
- keyword_query
- semantic_query
- filters
- constraints
- score_boost
- session_signals
- onboarding_signals

2. book: 추천 후보 도서 정보

[판정 기준]
- 현재 사용자 요청의 핵심 의도와 조건을 우선적으로 판단하세요.
- session_signals를 최우선 기준으로 판단하세요.
- onboarding_signals는 session_signals와 충돌하지 않는 경우에만 보조적으로 반영하세요.
- session_signals와 onboarding_signals가 충돌하면 session_signals를 우선하세요.
- 사용자 요청의 모든 표현이 책 정보에 직접 등장할 필요는 없습니다.
- 제목, 소개, 목차, 카테고리, 리뷰 등을 종합하여 의미적 적합성을 판단하세요.
- 책이 제공하는 독서 경험, 분위기, 주제, 대상 독자가 사용자 요청과 충분히 일치하면 높은 점수를 부여하세요.
- 일부 키워드만 우연히 일치하거나, 장르/주제/대상/분위기가 실제로 맞지 않으면 낮은 점수를 부여하세요.
- filters, constraints는 명시 조건으로 보고 우선 고려하세요.
- soft 조건은 약간 벗어나도 핵심 요청 적합성이 높으면 높은 점수를 부여할 수 있습니다.
- 정보가 부족하여 핵심 조건 충족 여부를 판단할 수 없으면 낮은 점수를 부여하세요.
- 요청과 직접 관련 없는 책이 단순히 키워드만 포함한 경우에는 낮은 점수를 부여하세요.
- retrieval rank, score, source 정보는 판단에 사용하지 마세요.
- 반드시 JSON만 출력하세요. 설명 문장, markdown, 코드블록은 출력하지 마세요.

[Grade 기준]
3 = 매우 적합. 현재 요청의 핵심 의도와 조건에 잘 부합함.
2 = 적합. 핵심 의도는 맞지만 일부 조건이 약함.
1 = 부분 관련. 관련성은 있으나 정답 도서로 보기에는 약함.
0 = 부적합. 현재 요청의 핵심 의도와 맞지 않음.

출력 형식:
{
  "relevance_grade": 0 또는 1 또는 2 또는 3,
  "reason": "판정 근거를 간략히 설명하는 텍스트"
}
"""

In [37]:
EXCLUDE_KEYS = {
    "retrieval_info",
    "rank",
    "score",
    "source",
    "retrieval_source",
    "retrieval_rank",
    "retrieval_score",
    "bm25_score",
    "dense_score",
    "hybrid_score",
    "main_score",
    "onboarding_score",
    "disliked_penalty",
    "disliked_similarity",
    # 기존 평가 결과 필드도 제거 (LLM 편향 방지)
    "relevance_grade",
    "binary_label",
    "label_status",
    "llm_raw_output",
    "llm_error",
    "query_id",
}


def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream"
    }


def make_book_for_judge(book):
    """
    LLM judge 입력에서 retrieval 관련 정보 및 기존 평가 필드를 제거한다.
    retrieval_info는 최종 저장 데이터에는 유지한다.
    """
    return {
        k: v for k, v in book.items()
        if k not in EXCLUDE_KEYS and not str(k).startswith("retrieval_")
    }


def parse_hcx_response(text):
    """
    HCX 응답이 SSE 형식 또는 일반 JSON 형식으로 올 수 있어 둘 다 처리한다.
    """
    # 1. SSE 형식 처리
    if "event:" in text:
        last_data = None

        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = None
            data_text = None

            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            data = json.loads(last_data)
            return data["message"]["content"]

    # 2. 일반 JSON 응답 처리
    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_grade(llm_text):
    """
    파싱 실패를 grade=0으로 처리하지 않는다.
    실패 시 relevance_grade=None, label_status='parse_failed'로 반환한다.
    """
    try:
        obj = json.loads(llm_text)
        grade = obj.get("relevance_grade", obj.get("grade", obj.get("label")))

        if grade is None:
            return None, "parse_failed", "missing relevance_grade"

        grade = int(grade)

        if grade not in [0, 1, 2, 3]:
            return None, "parse_failed", f"invalid relevance_grade: {grade}"

        return grade, "success", None

    except Exception as e:
        return None, "parse_failed", str(e)


def grade_to_binary(grade):
    if grade is None:
        return None
    return 1 if grade >= 2 else 0


def make_user_prompt(request_data, book):
    payload = {
        "request": request_data,
        "book": make_book_for_judge(book)
    }

    return json.dumps(payload, ensure_ascii=False, indent=2)

# 라벨링 함수

In [41]:
def blocking_label_one_book(request_data, book, max_retries=MAX_RETRIES):
    """
    동기 HTTP 요청 함수. asyncio.to_thread()로 thread pool에서 호출된다.
    기존 label_one_book과 동일한 로직.
    """
    query_id = request_data.get("query_id", "unknown")
    user_prompt = make_user_prompt(request_data, book)

    body = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 100,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=60
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                grade, label_status, parse_error = extract_grade(llm_text)
                binary_label = grade_to_binary(grade)
                # print(f"[성공] query_id={query_id}, book_id={book.get('id', 'unknown')}, grade={grade}, llm_raw_output={llm_text[:100]}...")
                return {
                    **book,
                    "query_id": query_id,
                    "relevance_grade": grade,
                    "binary_label": binary_label,
                    "label_status": label_status,
                    "llm_raw_output": llm_text,
                    "llm_error": parse_error
                }

            last_error = f"{res.status_code} / {res.text[:500]}"

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)

            # 400/401 등은 재시도해도 해결되지 않으므로 즉시 실패 처리
            return {
                **book,
                "query_id": query_id,
                "relevance_grade": None,
                "binary_label": None,
                "label_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": last_error
            }

        except Exception as e:
            last_error = str(e)

            if attempt == max_retries - 1:
                return {
                    **book,
                    "query_id": query_id,
                    "relevance_grade": None,
                    "binary_label": None,
                    "label_status": "api_failed",
                    "llm_raw_output": None,
                    "llm_error": last_error
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            # print(f"[재시도 {attempt + 1}/{max_retries}] {wait:.1f}초 대기 → {last_error}")
            time.sleep(wait)

In [39]:
async def label_one_book_async(request_data, book, semaphore, write_lock, jsonl_file, pbar):
    """
    비동기 래퍼:
    - semaphore로 동시 요청 수 제한
    - asyncio.to_thread로 blocking HTTP 호출을 thread pool에서 실행
    - write_lock으로 JSONL 파일 쓰기 직렬화
    """
    async with semaphore:
        try:
            result = await asyncio.to_thread(
                blocking_label_one_book,
                request_data,
                book
            )
        except Exception as e:
            query_id = request_data.get("query_id", "unknown")
            result = {
                **book,
                "query_id": query_id,
                "relevance_grade": None,
                "binary_label": None,
                "label_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": str(e)
            }

        async with write_lock:
            jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
            jsonl_file.flush()

        pbar.update(1)
        return result


async def main():
    # 1. 시나리오 로딩 → query_id별 request_data 매핑
    with open(SCENARIO_PATH, encoding="utf-8") as f:
        scenarios = json.load(f)

    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    print(f"시나리오 수: {len(query_map)}")

    # 2. goldset 로딩
    with open(INPUT_JSON, encoding="utf-8") as f:
        goldset = json.load(f)

    print(f"goldset 총 항목 수: {len(goldset):,}")

    # 3. 이미 처리된 (query_id, isbn) 로딩 (resume 지원)
    processed_keys: set = set()
    already_labeled: list = []

    if OUTPUT_JSONL.exists():
        with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    item = json.loads(line)
                    key = (item.get("query_id"), item.get("isbn"))
                    if key not in processed_keys:
                        processed_keys.add(key)
                        already_labeled.append(item)
                except json.JSONDecodeError:
                    continue

        print(f"이미 처리된 도서 수: {len(processed_keys):,}")

    # 4. 처리 대상 필터링
    to_process = [
        entry for entry in goldset
        if (entry.get("query_id"), entry.get("isbn")) not in processed_keys
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"처리 대상: {len(to_process):,}개")

    if not to_process:
        print("처리할 항목이 없습니다.")
        return already_labeled

    # 5. 비동기 실행
    semaphore  = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar       = tqdm(total=len(to_process), desc="재평가 중", unit="book")

    with OUTPUT_JSONL.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            label_one_book_async(
                query_map[entry["query_id"]],
                entry,
                semaphore,
                write_lock,
                jsonl_file,
                pbar
            )
            for entry in to_process
            if entry.get("query_id") in query_map
        ]

        new_results = await asyncio.gather(*tasks)

    pbar.close()

    # 6. 최종 JSON 저장 (기존 + 신규 병합, goldset 원본 순서 기준 정렬)
    all_results = already_labeled + list(new_results)

    order = {(e.get("query_id"), e.get("isbn")): i for i, e in enumerate(goldset)}
    all_results.sort(key=lambda x: order.get((x.get("query_id"), x.get("isbn")), 99999))

    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\nJSONL 저장 완료: {OUTPUT_JSONL}")
    print(f"JSON  저장 완료: {OUTPUT_JSON}")

    return all_results

# 실행

> **테스트**: Cell 2의 `LIMIT = 10` 으로 설정 후 실행  
> **전체**: `LIMIT = None` 으로 설정 후 실행

In [42]:
import nest_asyncio
nest_asyncio.apply()

results = asyncio.run(main())

시나리오 수: 21
goldset 총 항목 수: 477
이미 처리된 도서 수: 273
처리 대상: 204개


재평가 중: 100%|██████████| 204/204 [06:42<00:00,  1.97s/book]


JSONL 저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_reeval3.jsonl
JSON  저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_reeval3.json


# 결과 확인

In [43]:
# 통계
stats      = defaultdict(lambda: defaultdict(int))
fail_stats = defaultdict(int)

for item in results:
    qid    = item.get("query_id", "unknown")
    grade  = item.get("relevance_grade")
    status = item.get("label_status")

    if status == "success":
        stats[qid][grade] += 1
    else:
        fail_stats[qid] += 1

total_success = sum(sum(g.values()) for g in stats.values())
total_failed  = sum(fail_stats.values())

print(f"{'='*68}")
print(f"재평가 완료")
print(f"{'='*68}")
print(f"총 항목: {len(results):,}")
print(f"성공:    {total_success:,} ({total_success/len(results)*100:.1f}%)")
print(f"실패:    {total_failed:,} ({total_failed/len(results)*100:.1f}%)")
print()

print(f"[query_id별 relevance_grade 분포]")
print(f"{'query_id':>10}  {'grade 0':>7}  {'grade 1':>7}  {'grade 2':>7}  {'grade 3':>7}  {'failed':>7}  {'total':>7}")
print("-" * 68)

all_qids = sorted(set(list(stats.keys()) + list(fail_stats.keys())))
for qid in all_qids:
    g      = stats[qid]
    failed = fail_stats[qid]
    total  = sum(g.values()) + failed
    print(f"{qid:>10}  {g.get(0,0):>7}  {g.get(1,0):>7}  {g.get(2,0):>7}  {g.get(3,0):>7}  {failed:>7}  {total:>7}")

재평가 완료
총 항목: 477
성공:    359 (75.3%)
실패:    118 (24.7%)

[query_id별 relevance_grade 분포]
  query_id  grade 0  grade 1  grade 2  grade 3   failed    total
--------------------------------------------------------------------
       S01        5       47        4        1       18       75
       S02       10       49        1        0       23       83
       S03       21       29        3        0       17       70
       S08       10       55        4        1       16       86
       S15       12       41        0        0       27       80
       S17        7       30       18       11       17       83


In [44]:
df = pd.DataFrame(results)

status_dist = df["label_status"].value_counts(dropna=False).reset_index()
status_dist.columns = ["label_status", "count"]
status_dist["ratio"] = status_dist["count"] / len(df)

print(f"전체 라벨링 결과 수: {len(df):,}")
status_dist

전체 라벨링 결과 수: 477


,label_status,count,ratio
0,success,359,0.752621
1,api_failed,118,0.247379


In [45]:
success_df = df[df["label_status"] == "success"].copy()

grade_dist = (
    success_df["relevance_grade"]
    .value_counts()
    .sort_index()
    .reset_index()
)
grade_dist.columns = ["relevance_grade", "count"]
grade_dist["ratio"] = grade_dist["count"] / len(success_df)

print(f"성공 라벨 수: {len(success_df):,}")
print(f"실패/제외 수: {len(df) - len(success_df):,}")
grade_dist

성공 라벨 수: 359
실패/제외 수: 118


,relevance_grade,count,ratio
0,0.0,65,0.181058
1,1.0,251,0.699164
2,2.0,30,0.083565
3,3.0,13,0.036212


In [46]:
async def retry_api_failed():
    # 1. 결과 JSON 로딩
    if not OUTPUT_JSON.exists():
        print(f"파일 없음: {OUTPUT_JSON}\nmain() 을 먼저 실행하세요.")
        return []

    with OUTPUT_JSON.open("r", encoding="utf-8") as f:
        all_results = json.load(f)

    # 2. 시나리오 → query_map
    with open(SCENARIO_PATH, encoding="utf-8") as f:
        scenarios = json.load(f)

    query_map = {}
    for scenario in scenarios:
        rag_query = scenario["turns"][-1]["rag_query"].copy()
        rag_query["query_id"] = scenario["scenario_id"]
        query_map[scenario["scenario_id"]] = rag_query

    # 3. api_failed 항목 필터링
    to_retry = [e for e in all_results if e.get("label_status") == "api_failed"]
    print(f"전체: {len(all_results):,}개  |  재시도 대상(api_failed): {len(to_retry):,}개")

    if not to_retry:
        print("재시도할 항목이 없습니다.")
        return all_results

    # 4. 비동기 재호출
    semaphore = asyncio.Semaphore(CONCURRENCY)
    pbar      = tqdm(total=len(to_retry), desc="재시도 중", unit="book")

    async def retry_one(entry):
        query_id = entry.get("query_id", "unknown")
        if query_id not in query_map:
            pbar.update(1)
            return entry
        async with semaphore:
            try:
                result = await asyncio.to_thread(
                    blocking_label_one_book,
                    query_map[query_id],
                    entry
                )
            except Exception as e:
                result = {
                    **entry,
                    "label_status": "api_failed",
                    "llm_error": str(e)
                }
            pbar.update(1)
            return result

    new_results = await asyncio.gather(*[retry_one(e) for e in to_retry])
    pbar.close()

    # 5. 기존 결과에 재시도 결과 병합 (key: query_id + isbn)
    key_to_new = {(r.get("query_id"), r.get("isbn")): r for r in new_results}
    updated = [
        key_to_new.get((e.get("query_id"), e.get("isbn")), e)
        for e in all_results
    ]

    # 6. JSON 저장
    with OUTPUT_JSON.open("w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    # 7. 결과 요약
    now_success  = sum(1 for r in new_results if r.get("label_status") == "success")
    still_failed = sum(1 for r in new_results if r.get("label_status") == "api_failed")
    parse_failed = sum(1 for r in new_results if r.get("label_status") == "parse_failed")

    print(f"\n재시도 결과 ({len(to_retry)}개)")
    print(f"  성공:       {now_success:,}")
    print(f"  parse_failed: {parse_failed:,}")
    print(f"  api_failed:  {still_failed:,}")
    print(f"\nJSON 저장 완료: {OUTPUT_JSON}")

    return updated

In [50]:
results = await retry_api_failed()

전체: 477개  |  재시도 대상(api_failed): 5개


재시도 중:   0%|          | 0/5 [00:00<?, ?book/s]

재시도 중: 100%|██████████| 5/5 [00:08<00:00,  1.76s/book]


재시도 결과 (5개)
  성공:       5
  parse_failed: 0
  api_failed:  0

JSON 저장 완료: /home/ubuntu/work/somin/evaluation/dataset/goldset_s01_s02_s03_s08_s15_s17_reeval3.json


In [51]:
import json
from collections import Counter
with open(OUTPUT_JSON) as f:
    d = json.load(f)
print(Counter(r["label_status"] for r in d))


Counter({'success': 477})
